# Flavor Expansion Walkthrough

This notebook demonstrates the FeynRules-style model workflow with flavor-index expansion.

**Recommended workflow**

1. declare indices in Spenso (`flavor_index`, `COLOR_FUND_INDEX`, …)
2. declare gauge representations and `GaugeGroup(...)`
3. declare fields with `Field(...)`, `dirac_field(...)`, or `scalar_field(...)`; pass `class_members=(...)` and `flavor_index=...` for FeynRules-style flavor classes
4. declare parameters with `Parameter(...)`
5. build the model with `Model(gauge_groups=..., fields=..., parameters=..., lagrangian_decl=...)`
6. extract Feynman rules with `model.feynman_rule(..., flavor_expand=...)`

**What the examples below cover**

- explicit `IndexRole.FLAVOR` metadata and `Field`-level `class_members` / `flavor_index`
- compact vs expanded vertex signatures and rules
- two-index Yukawa matrices and off-diagonal zero `components`
- one-index diagonal tensors with `allow_summation=True`
- color indices preserved under flavor expansion
- shared and independent flavor labels across field classes
- selected flavor expansion (`flavor_expand=Generation` or an iterable)
- representative validation errors


In [1]:
import sys
from pathlib import Path
from fractions import Fraction

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from symbolica import S
from symbolica.community.spenso import Representation

from model import (
    COLOR_FUND_INDEX,
    Field,
    GaugeGroup,
    GaugeRepresentation,
    IndexRole,
    IndexType,
    Model,
    Parameter,
    SPINOR_INDEX,
    dirac_field,
    flavor_index,
    scalar_field,
)
from symbolic.spenso_structures import gauge_generator, structure_constant

print("Python:", sys.executable)
print("Repository root:", ROOT)


Python: /Users/rems/Documents/MScThesis/.venv/bin/python
Repository root: /Users/rems/Documents/MScThesis


In [2]:
def show(title, result):
  """Print vertex signatures and rules in a notebook-friendly format."""
  print(title)
  if isinstance(result, dict):
    print(f"{len(result)} vertex signature(s)\n")
    for signature, expression in result.items():
      print("Vertex:", signature)
      print("Rule:", expression)
      print()
  else:
    print(result)
    print()


def off_diagonal_zero_components(index):
  """Build {(i,j): 0} for all i != j — useful for diagonal Yukawa matrices."""
  return {
    (row, col): 0
    for row in range(1, index.dimension + 1)
    for col in range(1, index.dimension + 1)
    if row != col
  }


## Example 1 — Charged leptons: indices, fields, and compact vs expanded rules

Start with a flavor index, declare a generic lepton class with `dirac_field(..., class_members=(...), flavor_index=...)` — the same convention as FeynRules `ClassMembers` / `FlavorIndex` — then build a `Model` and call `model.feynman_rule(...)` directly. The members are auto-built with the same metadata as the class, minus the flavor-index slot, and are reachable via `field.class_members`.


In [3]:
# 1. Indices
Generation = flavor_index("Generation", 3, prefix="f")

# 2. Fields — class declaration with ClassMembers-like names
l = dirac_field(
    "l",
    class_members=("e", "mu", "ta"),
    indices=(Generation,),
    flavor_index=Generation,
    quantum_numbers={"Q": -1, "LeptonNumber": 1},
)
e, mu, ta = l.class_members
Phi = scalar_field("Phi")

print("flavor.role:", Generation.role)
print("flavor.dimension:", Generation.dimension)
print("l.indices:", [index.name for index in l.indices])
print("l.flavor_index_slot():", l.flavor_index_slot())
print("members:", [member.name for member in l.class_members])
print("member indices:", {member.name: [index.name for index in member.indices] for member in l.class_members})
print("mapping:", {k: l.class_member_for(k).name for k in range(1, Generation.dimension + 1)})

# 3. Model
f = S("f")
lepton_model = Model(
    name="Charged leptons",
    fields=(l, Phi),
    lagrangian_decl=S("lam") * l.bar(f) * l(f) * Phi,
)

print("\ncompact signatures:", [s.names for s in lepton_model.vertex_signatures(flavor_expand=False)])
print("expanded signatures:", [s.names for s in lepton_model.vertex_signatures(flavor_expand=True)])


flavor.role: IndexRole.FLAVOR
flavor.dimension: 3
l.indices: ['Generation', 'Spinor']
l.flavor_index_slot(): 0
members: ['e', 'mu', 'ta']
member indices: {'e': ['Spinor'], 'mu': ['Spinor'], 'ta': ['Spinor']}
mapping: {1: 'e', 2: 'mu', 3: 'ta'}

compact signatures: [('l.bar', 'l', 'Phi')]
expanded signatures: [('e.bar', 'e', 'Phi'), ('mu.bar', 'mu', 'Phi'), ('ta.bar', 'ta', 'Phi')]


The compact rule keeps the generic class field and a flavor identity tensor. After `flavor_expand=True`, only diagonal member vertices survive.


In [4]:
print("compact rule l.bar, l, Phi")
print(lepton_model.feynman_rule(l.bar, l, Phi, simplify=True))
print()
print("expanded rule e.bar, e, Phi")
print(lepton_model.feynman_rule(e.bar, e, Phi, simplify=True, flavor_expand=True))
print()
print("off-diagonal e.bar, mu, Phi is absent")
try:
    print(lepton_model.feynman_rule(e.bar, mu, Phi, simplify=True, flavor_expand=True))
except Exception as exc:
    print(type(exc).__name__ + ":", exc)


compact rule l.bar, l, Phi
1𝑖*lam*g(bis(4, i1),bis(4, i2))*g(cof(3, f1),cof(3, f2))

expanded rule e.bar, e, Phi
1𝑖*lam*g(bis(4, i1),bis(4, i2))

off-diagonal e.bar, mu, Phi is absent
ValueError: No matching interaction terms for: e.bar, mu, Phi.
Available signatures:
  - e.bar, e, Phi
  - mu.bar, mu, Phi
  - ta.bar, ta, Phi


## Example 2 — SM-like quark sector with gauge group, color, and diagonal Yukawa

Declare indices, a gauge representation, an `SU(3)` gauge group, up/down quark classes, and a diagonal Yukawa matrix with off-diagonal `components={(i,j): 0}`.


In [5]:
# 1. Indices
Colour = COLOR_FUND_INDEX

# 2. Gauge representation and group
colour_fund = GaugeRepresentation(
    index=Colour,
    generator_builder=gauge_generator,
    name="fund",
)
SU3C = GaugeGroup(
    name="SU3C",
    abelian=False,
    coupling=S("gs"),
    gauge_boson="G",
    structure_constant=structure_constant,
    representations=(colour_fund,),
)

# 3. Fields — flavor-generic up/down quark classes
uq = dirac_field(
    "uq",
    class_members=("u", "c", "t"),
    indices=(Generation, Colour),
    flavor_index=Generation,
    quantum_numbers={"Q": Fraction(2, 3)},
)
u, c, t = uq.class_members
dq = dirac_field(
    "dq",
    class_members=("d", "s", "b"),
    indices=(Generation, Colour),
    flavor_index=Generation,
    quantum_numbers={"Q": Fraction(-1, 3)},
)
d, s, b = dq.class_members

# 4. Parameters
gQ = Parameter("gQ")
yu = Parameter(
    "yu",
    indices=(Generation, Generation),
    components=off_diagonal_zero_components(Generation),
)

# 5. Model
f, h, colour = S("f", "h", "c")
quark_model = Model(
    name="Quark Yukawa demo",
    gauge_groups=(SU3C,),
    fields=(uq, dq, Phi),
    parameters=(gQ, yu),
    lagrangian_decl=(
        gQ * uq.bar(f, colour) * uq(f, colour) * Phi
        + yu(f, h) * uq.bar(f, colour) * uq(h, colour) * Phi
    ),
)

print("compact signatures:")
for signature in quark_model.vertex_signatures(flavor_expand=False):
    print(" ", signature.names)

print("\nexpanded signatures (Generation):")
for signature in quark_model.vertex_signatures(flavor_expand=Generation):
    print(" ", signature.names)

print("\nΓ(u.bar, u, Phi):")
print(quark_model.feynman_rule(u.bar, u, Phi, simplify=True, include_delta=True, flavor_expand=Generation))

print("\nΓ(t.bar, t, Phi):")
print(quark_model.feynman_rule(t.bar, t, Phi, simplify=True, include_delta=True, flavor_expand=Generation))

print("\noff-diagonal u.bar, c, Phi removed by components={(i,j): 0}:")
try:
    print(quark_model.feynman_rule(u.bar, c, Phi, simplify=True, include_delta=True, flavor_expand=Generation))
except Exception as exc:
    print(type(exc).__name__ + ":", exc)


compact signatures:
  ('uq.bar', 'uq', 'Phi')

expanded signatures (Generation):
  ('c.bar', 'c', 'Phi')
  ('t.bar', 't', 'Phi')
  ('u.bar', 'u', 'Phi')

Γ(u.bar, u, Phi):
1𝑖*gQ*(2*𝜋)^d*g(bis(4, i1),bis(4, i2))*g(cof(3, c1),cof(3, c2))*Delta(q1+q2+q3)+1𝑖*(2*𝜋)^d*g(bis(4, i1),bis(4, i2))*g(cof(3, c1),cof(3, c2))*Delta(q1+q2+q3)*yu(1,1)

Γ(t.bar, t, Phi):
1𝑖*gQ*(2*𝜋)^d*g(bis(4, i1),bis(4, i2))*g(cof(3, c1),cof(3, c2))*Delta(q1+q2+q3)+1𝑖*(2*𝜋)^d*g(bis(4, i1),bis(4, i2))*g(cof(3, c1),cof(3, c2))*Delta(q1+q2+q3)*yu(3,3)

off-diagonal u.bar, c, Phi removed by components={(i,j): 0}:
ValueError: No matching interaction terms for: u.bar, c, Phi.
Available signatures:
  - c.bar, c, Phi
  - t.bar, t, Phi
  - u.bar, u, Phi


## Example 3 — Non-diagonal two-index flavor tensor

Two independent flavor labels in one term expand to all member pairs.


In [6]:
f_mix, g_mix = S("fMix", "gMix")
Y = Parameter("Y", indices=(Generation, Generation))

mixed_model = Model(
    name="Lepton mixing",
    fields=(l, Phi),
    parameters=(Y,),
    lagrangian_decl=S("gY") * l.bar(f_mix) * Y(f_mix, g_mix) * l(g_mix) * Phi,
)

print("expanded signatures for Y[f, g]:")
print([s.names for s in mixed_model.vertex_signatures(flavor_expand=True)])
print()
print("e.bar, mu, Phi")
print(mixed_model.feynman_rule(e.bar, mu, Phi, simplify=True, flavor_expand=True))
print()
print("mu.bar, e, Phi")
print(mixed_model.feynman_rule(mu.bar, e, Phi, simplify=True, flavor_expand=True))
print()
print("ta.bar, ta, Phi")
print(mixed_model.feynman_rule(ta.bar, ta, Phi, simplify=True, flavor_expand=True))


expanded signatures for Y[f, g]:
[('e.bar', 'e', 'Phi'), ('e.bar', 'mu', 'Phi'), ('e.bar', 'ta', 'Phi'), ('mu.bar', 'e', 'Phi'), ('mu.bar', 'mu', 'Phi'), ('mu.bar', 'ta', 'Phi'), ('ta.bar', 'e', 'Phi'), ('ta.bar', 'mu', 'Phi'), ('ta.bar', 'ta', 'Phi')]

e.bar, mu, Phi
1𝑖*gY*g(bis(4, i1),bis(4, i2))*Y(1,2)

mu.bar, e, Phi
1𝑖*gY*g(bis(4, i1),bis(4, i2))*Y(2,1)

ta.bar, ta, Phi
1𝑖*gY*g(bis(4, i1),bis(4, i2))*Y(3,3)


## Example 4 — One-index diagonal tensor and `allow_summation=True`

The shorthand `y[f] l.bar[f] l[f] Phi` is rejected unless the parameter explicitly opts in.


In [7]:
y_bad = Parameter("yBad", indices=(Generation,))
bad_model = Model(
    fields=(l, Phi),
    parameters=(y_bad,),
    lagrangian_decl=y_bad(f) * l.bar(f) * l(f) * Phi,
)

print("without allow_summation:")
try:
    bad_model.vertex_signatures(flavor_expand=True)
except Exception as exc:
    print(type(exc).__name__ + ":", exc)
print()

y_diag = Parameter("yDiag", indices=(Generation,), allow_summation=True)
diag_model = Model(
    fields=(l, Phi),
    parameters=(y_diag,),
    lagrangian_decl=y_diag(f) * l.bar(f) * l(f) * Phi,
)

print("diagonal y[f] signatures:", [s.names for s in diag_model.vertex_signatures(flavor_expand=True)])
print()
print("mu.bar, mu, Phi")
print(diag_model.feynman_rule(mu.bar, mu, Phi, simplify=True, include_delta=True, flavor_expand=True))


without allow_summation:
ValueError: Parameter 'yBad' uses flavor label 'f' more than twice in one term. Mark the parameter with allow_summation=True to use this diagonal shorthand.

diagonal y[f] signatures: [('e.bar', 'e', 'Phi'), ('mu.bar', 'mu', 'Phi'), ('ta.bar', 'ta', 'Phi')]

mu.bar, mu, Phi
1𝑖*(2*𝜋)^d*g(bis(4, i1),bis(4, i2))*Delta(q1+q2+q3)*yDiag(2)


## Example 5 — Zero tensor components are dropped

When all off-diagonal `Ydiag(i, j)` entries are declared zero, only the three diagonal vertices survive after expansion.


In [8]:
Ydiag = Parameter(
    "Ydiag",
    indices=(Generation, Generation),
    components=off_diagonal_zero_components(Generation),
)

zero_model = Model(
    fields=(l, Phi),
    parameters=(Ydiag,),
    lagrangian_decl=S("gDiag") * l.bar(f_mix) * Ydiag(f_mix, g_mix) * l(g_mix) * Phi,
)

print("only diagonal signatures remain:", [s.names for s in zero_model.vertex_signatures(flavor_expand=True)])
print()
show("compact Feynman rules", zero_model.feynman_rules(flavor_expand=False))
show("expanded Feynman rules", zero_model.feynman_rules(flavor_expand=True))
print("e.bar, mu, Phi was dropped")
try:
    print(zero_model.feynman_rule(e.bar, mu, Phi, flavor_expand=True))
except Exception as exc:
    print(type(exc).__name__ + ":", exc)


only diagonal signatures remain: [('e.bar', 'e', 'Phi'), ('mu.bar', 'mu', 'Phi'), ('ta.bar', 'ta', 'Phi')]

compact Feynman rules
1 vertex signature(s)

Vertex: ('l.bar', 'l', 'Phi')
Rule: 1𝑖*gDiag*g(bis(4, i1),bis(4, i2))*Ydiag(f1,f2)

expanded Feynman rules
3 vertex signature(s)

Vertex: ('e.bar', 'e', 'Phi')
Rule: 1𝑖*gDiag*g(bis(4, i1),bis(4, i2))*Ydiag(1,1)

Vertex: ('mu.bar', 'mu', 'Phi')
Rule: 1𝑖*gDiag*g(bis(4, i1),bis(4, i2))*Ydiag(2,2)

Vertex: ('ta.bar', 'ta', 'Phi')
Rule: 1𝑖*gDiag*g(bis(4, i1),bis(4, i2))*Ydiag(3,3)

e.bar, mu, Phi was dropped
ValueError: No matching interaction terms for: e.bar, mu, Phi.
Available signatures:
  - e.bar, e, Phi
  - mu.bar, mu, Phi
  - ta.bar, ta, Phi


## Example 6 — Color indices are not flavor-expanded

A quark class carries `(Generation, Colour, Spinor)`. Expansion removes only the flavor slot and keeps the color label symbolic.


In [9]:
# Smaller 2-generation example for a compact color check
Gen2 = flavor_index("Gen2", 2, prefix="fq")
fq, c = S("fq", "c")

q2 = dirac_field(
    "q",
    class_members=("d", "u"),
    indices=(Gen2, Colour),
    flavor_index=Gen2,
)
d, u = q2.class_members
PhiQ = scalar_field("PhiQ")

color_model = Model(
    gauge_groups=(SU3C,),
    fields=(q2, PhiQ),
    lagrangian_decl=S("gQ") * q2.bar(fq, c) * q2(fq, c) * PhiQ,
)

print("expanded signatures:", [s.names for s in color_model.vertex_signatures(flavor_expand=True)])
print()
print("d.bar, d, PhiQ still carries a color identity tensor")
print(color_model.feynman_rule(d.bar, d, PhiQ, simplify=True, include_delta=True, flavor_expand=True))


expanded signatures: [('d.bar', 'd', 'PhiQ'), ('u.bar', 'u', 'PhiQ')]

d.bar, d, PhiQ still carries a color identity tensor
1𝑖*gQ*(2*𝜋)^d*g(bis(4, i1),bis(4, i2))*g(cof(3, c1),cof(3, c2))*Delta(q1+q2+q3)


## Example 7 — Shared flavor label across two field classes

The same `Generation` label synchronizes left- and right-handed lepton classes.


In [10]:
lL = dirac_field(
    "lL",
    class_members=("eL", "muL", "taL"),
    indices=(Generation,),
    flavor_index=Generation,
)
eL, muL, taL = lL.class_members
lR = dirac_field(
    "lR",
    class_members=("eR", "muR", "taR"),
    indices=(Generation,),
    flavor_index=Generation,
)
eR, muR, taR = lR.class_members
yLR = Parameter("yLR", indices=(Generation,), allow_summation=True)

shared_model = Model(
    fields=(lL, lR, Phi),
    parameters=(yLR,),
    lagrangian_decl=yLR(f) * lL.bar(f) * lR(f) * Phi,
)

print("shared flavor label across lL and lR:", [s.names for s in shared_model.vertex_signatures(flavor_expand=True)])
print()
print("taL.bar, taR, Phi")
print(shared_model.feynman_rule(taL.bar, taR, Phi, simplify=True, include_delta=True, flavor_expand=True))


shared flavor label across lL and lR: [('eL.bar', 'eR', 'Phi'), ('muL.bar', 'muR', 'Phi'), ('taL.bar', 'taR', 'Phi')]

taL.bar, taR, Phi
1𝑖*(2*𝜋)^d*g(bis(4, i1),bis(4, i2))*Delta(q1+q2+q3)*yLR(3)


## Example 8 — Independent flavor labels: CKM-like up–down mixing

Two different field classes with independent flavor labels `f` and `h` expand to all `(up, down)` member pairs.


In [11]:
W = scalar_field("W")
V = Parameter("V", indices=(Generation, Generation))

mixing_model = Model(
    gauge_groups=(SU3C,),
    fields=(uq, dq, W),
    parameters=(V,),
    lagrangian_decl=uq.bar(f, colour) * V(f, h) * dq(h, colour) * W,
)

print("independent uq[f] and dq[h] labels:")
print([s.names for s in mixing_model.vertex_signatures(flavor_expand=True)])
print()
print("u.bar, s, W")
print(mixing_model.feynman_rule(u.bar, s, W, simplify=True, include_delta=True, flavor_expand=True))
print()
print("t.bar, b, W")
print(mixing_model.feynman_rule(t.bar, b, W, simplify=True, include_delta=True, flavor_expand=True))


independent uq[f] and dq[h] labels:
[('c.bar', 'b', 'W'), ('c.bar', 'd', 'W'), ('c.bar', 's', 'W'), ('t.bar', 'b', 'W'), ('t.bar', 'd', 'W'), ('t.bar', 's', 'W'), ('u.bar', 'b', 'W'), ('u.bar', 'd', 'W'), ('u.bar', 's', 'W')]

u.bar, s, W
1𝑖*(2*𝜋)^d*g(bis(4, i1),bis(4, i2))*g(cof(3, c1),cof(3, c2))*Delta(q1+q2+q3)*V(1,2)

t.bar, b, W
1𝑖*(2*𝜋)^d*g(bis(4, i1),bis(4, i2))*g(cof(3, c1),cof(3, c2))*Delta(q1+q2+q3)*V(3,3)


## Example 9 — Selected flavor expansion

`flavor_expand` can target one index (`flavor_expand=Generation`) or several (`flavor_expand=(Generation, SU2D)`), instead of expanding every flavor index at once.


In [12]:
SU2D = flavor_index("SU2D", 2, prefix="d")
chi = dirac_field(
    "chi",
    class_members=("chi1", "chi2"),
    indices=(SU2D,),
    flavor_index=SU2D,
)
chi1, chi2 = chi.class_members
d_sel, c_sel = S("d", "c")

selected_model = Model(
    fields=(uq, chi, Phi),
    lagrangian_decl=S("gSel") * uq.bar(f, c_sel) * chi(d_sel) * Phi,
)

print("expand Generation only:")
print([s.names for s in selected_model.vertex_signatures(flavor_expand=Generation)])
print()
print("expand SU2D only:")
print([s.names for s in selected_model.vertex_signatures(flavor_expand=SU2D)])
print()
print("expand both:")
print([s.names for s in selected_model.vertex_signatures(flavor_expand=(Generation, SU2D))])


expand Generation only:
[('c.bar', 'chi', 'Phi'), ('t.bar', 'chi', 'Phi'), ('u.bar', 'chi', 'Phi')]

expand SU2D only:
[('uq.bar', 'chi1', 'Phi'), ('uq.bar', 'chi2', 'Phi')]

expand both:
[('c.bar', 'chi1', 'Phi'), ('c.bar', 'chi2', 'Phi'), ('t.bar', 'chi1', 'Phi'), ('t.bar', 'chi2', 'Phi'), ('u.bar', 'chi1', 'Phi'), ('u.bar', 'chi2', 'Phi')]


## Example 10 — Representative validation errors

The metadata layer fails early when declarations are inconsistent.


In [13]:
print("wrong member count")
try:
    dirac_field(
        "BadPsi",
        class_members=("e", "mu"),
        indices=(Generation,),
        flavor_index=Generation,
    )
except Exception as exc:
    print(type(exc).__name__ + ":", exc)
print()

l_no_members = Field(
    "lNoMembers",
    spin=Fraction(1, 2),
    self_conjugate=False,
    symbol=S("l_no_members"),
    conjugate_symbol=S("l_no_members_bar"),
    indices=(Generation, SPINOR_INDEX),
    flavor_index=Generation,
)
missing_model = Model(
    fields=(l_no_members, Phi),
    lagrangian_decl=S("gMissing") * l_no_members.bar(f) * l_no_members(f) * Phi,
)
print("flavor expansion requested but no class members are defined")
try:
    missing_model.vertex_signatures(flavor_expand=True)
except Exception as exc:
    print(type(exc).__name__ + ":", exc)
print()

plain_index = IndexType(
    "PlainGeneration",
    Representation.cof(3),
    kind="plain_generation",
    dimension=3,
    prefix="p",
)
print("flavor_index must use role=IndexRole.FLAVOR")
try:
    Field(
        "WrongRole",
        spin=Fraction(1, 2),
        self_conjugate=False,
        symbol=S("wrongrole"),
        conjugate_symbol=S("wrongrolebar"),
        indices=(plain_index, SPINOR_INDEX),
        flavor_index=plain_index,
        class_members=(e, mu, ta),
    )
except Exception as exc:
    print(type(exc).__name__ + ":", exc)


wrong member count
ValueError: Field 'BadPsi' declares 2 class member(s), but flavor index 'Generation' has dimension 3.

flavor expansion requested but no class members are defined
ValueError: Cannot flavor-expand field 'lNoMembers' over index 'Generation' because no class members are defined.

flavor_index must use role=IndexRole.FLAVOR
ValueError: Field 'WrongRole' declares flavor_index='PlainGeneration', but that index type is not marked with role=IndexRole.FLAVOR.
